# Capstone Project
## Neural translation model
### Instructions

In this notebook, you will create a neural network that translates from English to German. You will use concepts from throughout this course, including building more flexible model architectures, freezing layers, data processing pipeline and sequence modelling.

This project is peer-assessed. Within this notebook you will find instructions in each section for how to complete the project. Pay close attention to the instructions as the peer review will be carried out according to a grading rubric that checks key parts of the project instructions. Feel free to add extra cells into the notebook as required.

### How to submit

When you have completed the Capstone project notebook, you will submit a pdf of the notebook for peer review. First ensure that the notebook has been fully executed from beginning to end, and all of the cell outputs are visible. This is important, as the grading rubric depends on the reviewer being able to view the outputs of your notebook. Save the notebook as a pdf (File -> Download as -> PDF via LaTeX). You should then submit this pdf for review.

### Let's get started!

We'll start by running some imports, and loading the dataset. For this project you are free to make further imports throughout the notebook as you wish. 

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import unicodedata
import re

![Flags overview image](data/germany_uk_flags.png)

For the capstone project, you will use a language dataset from http://www.manythings.org/anki/ to build a neural translation model. This dataset consists of over 200,000 pairs of sentences in English and German. In order to make the training quicker, we will restrict to our dataset to 20,000 pairs. Feel free to change this if you wish - the size of the dataset used is not part of the grading rubric.

Your goal is to develop a neural translation model from English to German, making use of a pre-trained English word embedding module.

In [ ]:
# Run this cell to load the dataset

NUM_EXAMPLES = 20000
data_examples = []
with open('data/deu.txt', 'r', encoding='utf8') as f:
    for line in f.readlines():
        if len(data_examples) < NUM_EXAMPLES:
            data_examples.append(line)
        else:
            break

In [ ]:
# These functions preprocess English and German sentences

def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"ü", 'ue', sentence)
    sentence = re.sub(r"ä", 'ae', sentence)
    sentence = re.sub(r"ö", 'oe', sentence)
    sentence = re.sub(r'ß', 'ss', sentence)
    
    sentence = unicode_to_ascii(sentence)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"[^a-z?.!,']+", " ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    
    return sentence.strip()

#### The custom translation model
The following is a schematic of the custom translation model architecture you will develop in this project.

![Model Schematic](data/neural_translation_model.png)

Key:
![Model key](data/neural_translation_model_key.png)

The custom model consists of an encoder RNN and a decoder RNN. The encoder takes words of an English sentence as input, and uses a pre-trained word embedding to embed the words into a 128-dimensional space. To indicate the end of the input sentence, a special end token (in the same 128-dimensional space) is passed in as an input. This token is a TensorFlow Variable that is learned in the training phase (unlike the pre-trained word embedding, which is frozen).

The decoder RNN takes the internal state of the encoder network as its initial state. A start token is passed in as the first input, which is embedded using a learned German word embedding. The decoder RNN then makes a prediction for the next German word, which during inference is then passed in as the following input, and this process is repeated until the special `<end>` token is emitted from the decoder.

## 1. Text preprocessing
* Create separate lists of English and German sentences, and preprocess them using the `preprocess_sentence` function provided for you above.
* Add a special `"<start>"` and `"<end>"` token to the beginning and end of every German sentence.
* Use the Tokenizer class from the `tf.keras.preprocessing.text` module to tokenize the German sentences, ensuring that no character filters are applied. _Hint: use the Tokenizer's "filter" keyword argument._
* Print out at least 5 randomly chosen examples of (preprocessed) English and German sentence pairs. For the German sentence, print out the text (with start and end tokens) as well as the tokenized sequence.
* Pad the end of the tokenized German sequences with zeros, and batch the complete set of sequences into one numpy array.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np
import random

In [ ]:
english_sentences = [preprocess_sentence(s.split('\t')[0]) for s in data_examples]
german_sentences = ["<start> " + preprocess_sentence(s.split('\t')[1]) + " <end>"  for s in data_examples]

In [ ]:
tokenizer = Tokenizer(num_words=None, 
                      filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n',
                      lower=True,
                      split=' ',
                      char_level=False,
                      oov_token='<UNK>',
                      document_count=0)

tokenizer.fit_on_texts(german_sentences)

In [ ]:
indices = random.sample(list(range(NUM_EXAMPLES)), 5)

for i in indices:
    print(i,english_sentences[i], german_sentences[i], tokenizer.texts_to_sequences([german_sentences[i]]))

In [ ]:
german_encoded = tokenizer.texts_to_sequences(german_sentences)

german_encoded = tf.keras.preprocessing.sequence.pad_sequences(german_encoded,
                                                           padding='post',
                                                           value=0)


In [ ]:
german_encoded

## 2. Prepare the data with tf.data.Dataset objects

#### Load the embedding layer
As part of the dataset preproceessing for this project, you will use a pre-trained English word embedding module from TensorFlow Hub. The URL for the module is https://tfhub.dev/google/tf2-preview/nnlm-en-dim128-with-normalization/1. This module has also been made available as a complete saved model in the folder `'./models/tf2-preview_nnlm-en-dim128_1'`. 

This embedding takes a batch of text tokens in a 1-D tensor of strings as input. It then embeds the separate tokens into a 128-dimensional space. 

The code to load and test the embedding layer is provided for you below.

**NB:** this model can also be used as a sentence embedding module. The module will process each token by removing punctuation and splitting on spaces. It then averages the word embeddings over a sentence to give a single embedding vector. However, we will use it only as a word embedding module, and will pass each word in the input sentence as a separate token.

In [ ]:
# Load embedding module from Tensorflow Hub

#embedding_layer = hub.KerasLayer("https://tfhub.dev/google/tf2-preview/nnlm-en-dim128/1", 
#                                 output_shape=[128], input_shape=[], dtype=tf.string)

embedding_layer = tf.keras.models.load_model('./models/tf2-preview_nnlm-en-dim128_1')
embedding_layer.trainable = False

In [ ]:
# Test the layer

embedding_layer(tf.constant(["these", "aren't", "the", "droids", "you're", "looking", "for"])).shape

You should now prepare the training and validation Datasets.

* Create a random training and validation set split of the data, reserving e.g. 20% of the data for validation (NB: each English dataset example is a single sentence string, and each German dataset example is a sequence of padded integer tokens).
* Load the training and validation sets into a tf.data.Dataset object, passing in a tuple of English and German data for both training and validation sets.
* Create a function to map over the datasets that splits each English sentence at spaces. Apply this function to both Dataset objects using the map method. _Hint: look at the tf.strings.split function._
* Create a function to map over the datasets that embeds each sequence of English words using the loaded embedding layer/model. Apply this function to both Dataset objects using the map method.
* Create a function to filter out dataset examples where the English sentence is more than 13 (embedded) tokens in length. Apply this function to both Dataset objects using the filter method.
* Create a function to map over the datasets that pads each English sequence of embeddings with some distinct padding value before the sequence, so that each sequence is length 13. Apply this function to both Dataset objects using the map method. _Hint: look at the tf.pad function. You can extract a Tensor shape using tf.shape; you might also find the tf.math.maximum function useful._
* Batch both training and validation Datasets with a batch size of 16.
* Print the `element_spec` property for the training and validation Datasets. 
* Using the Dataset `.take(1)` method, print the shape of the English data example from the training Dataset.
* Using the Dataset `.take(1)` method, print the German data example Tensor from the validation Dataset.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(german_encoded, english_sentences, test_size=0.2, random_state=42)

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

In [ ]:
def split_eng_spaces(ger_tokens,eng_sentence):
    return ger_tokens,tf.strings.split(eng_sentence, sep=" ")


In [ ]:
train_ds = train_ds.map(split_eng_spaces)
test_ds = test_ds.map(split_eng_spaces)

In [ ]:
def make_embeddings(x,y):
    s=embedding_layer(y)
    return x, s

In [ ]:
train_ds = train_ds.map(make_embeddings)
test_ds = test_ds.map(make_embeddings)

In [ ]:
def filter_by_size(x,y):
    return int(tf.shape(y)[0]) <= 13

In [ ]:
train_ds = train_ds.filter(filter_by_size)
test_ds = test_ds.filter(filter_by_size)

Create a function to map over the datasets that pads each English sequence of embeddings with some distinct padding value before the sequence, so that each sequence is length 13. Apply this function to both Dataset objects using the map method. Hint: look at the tf.pad function. You can extract a Tensor shape using tf.shape; you might also find the tf.math.maximum function useful.

In [ ]:
@tf.function
def make_eng_padding(x,y):
    size = tf.shape(y)[0]
    padding = [[13-size,0], [0,0]]
    return x, tf.pad(y, padding)


In [ ]:
train_ds = train_ds.map(make_eng_padding)
test_ds = test_ds.map(make_eng_padding)

In [ ]:
train_ds = train_ds.batch(16)
test_ds = test_ds.batch(16)

In [ ]:
print(train_ds.element_spec)
print(test_ds.element_spec)

In [ ]:
#Using the Dataset .take(1) method, print the shape of the English data example from the training Dataset.
print(list(train_ds.take(1))[0][1].shape)

#Using the Dataset .take(1) method, print the German data example Tensor from the validation Dataset.
print(list(train_ds.take(1))[0][0])

## 3. Create the custom layer
You will now create a custom layer to add the learned end token embedding to the encoder model:

![Encoder schematic](data/neural_translation_model_encoder.png)

You should now build the custom layer.
* Using layer subclassing, create a custom layer that takes a batch of English data examples from one of the Datasets, and adds a learned embedded ‘end’ token to the end of each sequence. 
* This layer should create a TensorFlow Variable (that will be learned during training) that is 128-dimensional (the size of the embedding space). _Hint: you may find it helpful in the call method to use the tf.tile function to replicate the end token embedding across every element in the batch._
* Using the Dataset `.take(1)` method, extract a batch of English data examples from the training Dataset and print the shape. Test the custom layer by calling the layer on the English data batch Tensor and print the resulting Tensor shape (the layer should increase the sequence length by one).

In [ ]:
from tensorflow.keras.layers import Layer

class AddEndLayer(Layer):

    def __init__(self, **kwargs):
        super(AddEndLayer, self).__init__(**kwargs)
        self.end = tf.Variable(initial_value=tf.ones((1,1,128)))
        
    def call(self, inputs):
        return tf.concat([inputs, tf.tile(self.end, [inputs.shape[0], 1, 1])], 1)

In [ ]:
add_end_layer = AddEndLayer()

for x,y in train_ds.take(1):
    print(y.shape)
    print(add_end_layer(y).shape)

## 4. Build the encoder network
The encoder network follows the schematic diagram above. You should now build the RNN encoder model.
* Using the functional API, build the encoder network according to the following spec:
    * The model will take a batch of sequences of embedded English words as input, as given by the Dataset objects.
    * The next layer in the encoder will be the custom layer you created previously, to add a learned end token embedding to the end of the English sequence.
    * This is followed by a Masking layer, with the `mask_value` set to the distinct padding value you used when you padded the English sequences with the Dataset preprocessing above.
    * The final layer is an LSTM layer with 512 units, which also returns the hidden and cell states.
    * The encoder is a multi-output model. There should be two output Tensors of this model: the hidden state and cell states of the LSTM layer. The output of the LSTM layer is unused.
* Using the Dataset `.take(1)` method, extract a batch of English data examples from the training Dataset and test the encoder model by calling it on the English data Tensor, and print the shape of the resulting Tensor outputs.
* Print the model summary for the encoder network.

In [ ]:
from tensorflow.keras.layers import Masking, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

In [ ]:
class Encoder(Model):

    def __init__(self, **kwargs):
        """
        The class initialiser should call the base class initialiser, passing any keyword
        arguments along. It should also create the layers of the network according to the
        above specification.
        """
        super(Encoder, self).__init__(**kwargs)
        self.add_end = AddEndLayer()
        self.masking = Masking(mask_value=0)
        self.lstm = LSTM(512, return_state=True)
                
    def call(self, inputs):
        """
        This method should contain the code for calling the layer according to the above
        specification, using the layer objects set up in the initialiser.
        """
        X = self.add_end(inputs)
        X = self.masking(X)
        something, h, c = self.lstm(X)
        return h, c
        

In [ ]:
encoder = Encoder()

for ger,eng in train_ds.take(2):
    print(eng.shape)
    res = encoder(eng)
    print(res[0].shape, res[1].shape)

In [ ]:
encoder.summary()

## 5. Build the decoder network
The decoder network follows the schematic diagram below. 

![Decoder schematic](data/neural_translation_model_decoder.png)

You should now build the RNN decoder model.
* Using Model subclassing, build the decoder network according to the following spec:
    * The initializer should create the following layers:
        * An Embedding layer with vocabulary size set to the number of unique German tokens, embedding dimension 128, and set to mask zero values in the input.
        * An LSTM layer with 512 units, that returns its hidden and cell states, and also returns sequences.
        * A Dense layer with number of units equal to the number of unique German tokens, and no activation function.
    * The call method should include the usual `inputs` argument, as well as the additional keyword arguments `hidden_state` and `cell_state`. The default value for these keyword arguments should be `None`.
    * The call method should pass the inputs through the Embedding layer, and then through the LSTM layer. If the `hidden_state` and `cell_state` arguments are provided, these should be used for the initial state of the LSTM layer. _Hint: use the_ `initial_state` _keyword argument when calling the LSTM layer on its input._
    * The call method should pass the LSTM output sequence through the Dense layer, and return the resulting Tensor, along with the hidden and cell states of the LSTM layer.
* Using the Dataset `.take(1)` method, extract a batch of English and German data examples from the training Dataset. Test the decoder model by first calling the encoder model on the English data Tensor to get the hidden and cell states, and then call the decoder model on the German data Tensor and hidden and cell states, and print the shape of the resulting decoder Tensor outputs.
* Print the model summary for the decoder network.

In [ ]:
class Decoder(Model):

    def __init__(self, **kwargs):
        super(Decoder, self).__init__(**kwargs)
        self.embedding = Embedding(input_dim=np.max(german_encoded)+1, output_dim=128, mask_zero=True)
        self.lstm = LSTM(512, return_state=True, return_sequences=True)
        self.dense = Dense(np.max(german_encoded)+1)
                
    def call(self, inputs, hidden_state=None, cell_state=None):
        #print(inputs[0])
        X = self.embedding(inputs)
        out, h, c = self.lstm(X, initial_state=[hidden_state, cell_state])
        #print(out.shape, h.shape, c.shape)
        out = self.dense(out)
        return out, h, c

In [ ]:
encoder = Encoder()
decoder = Decoder()

for g,e in train_ds.take(1):
    h, c = encoder(e)
    ger = decoder(g, h, c)
    print(ger[0].shape)

In [ ]:
decoder.summary()

## 6. Make a custom training loop
You should now write a custom training loop to train your custom neural translation model.
* Define a function that takes a Tensor batch of German data (as extracted from the training Dataset), and returns a tuple containing German inputs and outputs for the decoder model (refer to schematic diagram above).
* Define a function that computes the forward and backward pass for your translation model. This function should take an English input, German input and German output as arguments, and should do the following:
    * Pass the English input into the encoder, to get the hidden and cell states of the encoder LSTM.
    * These hidden and cell states are then passed into the decoder, along with the German inputs, which returns a sequence of outputs (the hidden and cell state outputs of the decoder LSTM are unused in this function).
    * The loss should then be computed between the decoder outputs and the German output function argument.
    * The function returns the loss and gradients with respect to the encoder and decoder’s trainable variables.
    * Decorate the function with @tf.function
* Define and run a custom training loop for a number of epochs (for you to choose) that does the following:
    * Iterates through the training dataset, and creates decoder inputs and outputs from the German sequences.
    * Updates the parameters of the translation model using the gradients of the function above and an optimizer object.
    * Every epoch, compute the validation loss on a number of batches from the validation and save the epoch training and validation losses.
* Plot the learning curves for loss vs epoch for both training and validation sets.

_Hint: This model is computationally demanding to train. The quality of the model or length of training is not a factor in the grading rubric. However, to obtain a better model we recommend using the GPU accelerator hardware on Colab._

In [ ]:
def german_fn(german_batch):
    # first we will exclude padding zeroes to improve training
    last_dim = german_batch.shape[1] - 1
    
    zeros = tf.zeros((german_batch.shape[0],), dtype=tf.dtypes.int32)
    while(tf.reduce_sum(tf.cast(tf.equal(german_batch[:,last_dim], zeros),tf.int32)) == german_batch.shape[0] and last_dim>0):
        last_dim = last_dim-1
    german_batch = german_batch[:,:last_dim]
    
    return german_batch[:,1:], german_batch[:,:-1] 

In [ ]:
scce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

@tf.function
def fb_pass(eng_in, ger_in):
    with tf.GradientTape() as tape:
        h,c = encoder(eng_in)
        ger_out, ger_in = german_fn(ger_in)
        ger_pred, _, _ = decoder(ger_in, h, c)
        loss = scce(ger_out, ger_pred)
        grads = tape.gradient(loss, [encoder.trainable_variables, decoder.trainable_variables])    

    return loss, grads


In [ ]:
@tf.function
def fb_pass_val(eng, ger):
        h,c = encoder(eng)
        ger_out, ger_in = german_fn(ger)
        ger_pred, _,_ = decoder(ger_in, h, c)
        loss = scce(ger_out, ger_pred) 
        return loss

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)

train_loss = []
val_loss = []

epochs=5

train_batches = 160 # to speed up
val_batches = 40 # to speed up

for epoch in range(epochs):
        print("Epoch:", epoch)
        for ger,eng in train_ds.take(train_batches).cache(): 
            if len(train_loss) % 100 ==0 :print("training batch no ",len(train_loss))
            loss, grads = fb_pass(eng, ger)
            opt.apply_gradients(zip(grads[0]+grads[1], encoder.trainable_variables + decoder.trainable_variables))
            print("training loss:", loss)
            train_loss.append(loss.numpy())

        for ger,eng in test_ds.take(val_batches).cache(): 
            loss = fb_pass_val(eng, ger)
            print("val loss:", loss)
            val_loss.append(loss.numpy())
            


In [ ]:
val_avg = []
for t in range(epochs):
    val_avg.append(np.mean(val_loss[t*val_batches: (t+1)*val_batches]))

train_x_axis = [x/train_batches for x in list(range(len(train_loss)))]

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.plot(range(1,epochs+1), val_avg, label = "validation")
plt.plot(train_x_axis, train_loss, label = "train")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

## 7. Use the model to translate
Now it's time to put your model into practice! You should run your translation for five randomly sampled English sentences from the dataset. For each sentence, the process is as follows:
* Preprocess and embed the English sentence according to the model requirements.
* Pass the embedded sentence through the encoder to get the encoder hidden and cell states.
* Starting with the special  `"<start>"` token, use this token and the final encoder hidden and cell states to get the one-step prediction from the decoder, as well as the decoder’s updated hidden and cell states.
* Create a loop to get the next step prediction and updated hidden and cell states from the decoder, using the most recent hidden and cell states. Terminate the loop when the `"<end>"` token is emitted, or when the sentence has reached a maximum length.
* Decode the output token sequence into German text and print the English text and the model's German translation.

In [ ]:
indices = random.sample(list(range(NUM_EXAMPLES)), 5)

for i in indices:
    print("next pair:")
    print("English:", english_sentences[i],"German:", german_sentences[i])
    
    # Preprocess and embed the English sentence according to the model requirements.
    eng = embedding_layer([english_sentences[i]])
    #print("eng embedding:", eng)
    size = tf.shape(eng)[0]
    padding = [[13-size,0], [0,0]]
    eng = tf.pad(eng, padding)
    eng = tf.expand_dims(eng, axis=0)
    #print("eng padded:", eng.shape)
    # Pass the embedded sentence through the encoder to get the encoder hidden and cell states.
    h,c = encoder(eng)
    # Starting with the special "<start>" token, use this token and the final encoder hidden and cell states to get the one-step prediction from the decoder, as well as the decoder’s updated hidden and cell states.
    german_end = ["<end>"]
    end_code = tokenizer.texts_to_sequences(german_end)
    #print("end_code", end_code)
    
    german_start = ["<start>"]
    current_code = tokenizer.texts_to_sequences(german_start)
    #print("current_code", current_code)    
    
    last = current_code
    
    while len(current_code[0]) <= 13 and last != end_code:
        #out, h, c = decoder.lstm(X, initial_state=[h, c])
    
        #german_encoded = tf.keras.preprocessing.sequence.pad_sequences(current_code,
        #                                                   padding='post',
        #                                                   value=0)
        current_code = tf.convert_to_tensor(current_code, dtype=tf.int64)
        #print("german_encoded", current_code, type(current_code))
        X = decoder.embedding(current_code)
        out, h, c = decoder.lstm(X, initial_state=[h, c])
        out = decoder.dense(out)
        #print("out shape", out.shape, out[:,-1,:].shape)
        last = tf.math.argmax(out[:,-1,:], axis=1)
        last = tf.expand_dims(last, axis=0)
        #print("last:", last)
        current_code = tf.concat([current_code, last], 1)
    #print("Sequence:", current_code)
    print("German model translation:", tokenizer.sequences_to_texts(current_code.numpy()))

    # Create a loop to get the next step prediction and updated hidden and cell states from the decoder, using the most recent hidden and cell states. Terminate the loop when the "<end>" token is emitted, or when the sentence has reached a maximum length.
    # Decode the output token sequence into German text and print the English text and the model's German translation.
